In [15]:
import math
from scipy.integrate import dblquad

In [16]:
class Piece:
    def __init__(self, nom, x, y, radius, length, mass, forme):
        self.nom = nom
        self.x = x
        self.y = y
        self.radius = radius
        self.length = length
        self.mass = mass
        self.forme = forme  # 'cylindrique', 'conique', 'demi-cylindrique'

In [17]:
diameter_int = 240

EngineBay       = Piece("EngineBay",       0,                                    0,              diameter_int/2, 995,  11.11, "cylindrique")
LOx             = Piece("LOx",             EngineBay.length,                     0,              diameter_int/2, 622,  7.4,   "cylindrique")
MidBay          = Piece("MidBay",          LOx.x + LOx.length,                   0,              diameter_int/2, 280,  3.13,  "cylindrique")
PressurantBay1  = Piece("PressurantBay1",  MidBay.x + MidBay.length,             0,              diameter_int/2, 280,  3.13,  "cylindrique")
ETH             = Piece("ETH",             PressurantBay1.x + PressurantBay1.length, 0,           diameter_int/2, 622,  7.4,   "cylindrique")
PressurantBay2  = Piece("PressurantBay2",  ETH.x + ETH.length,                   0,              diameter_int/2, 280,  3.13,  "cylindrique")
AVBay           = Piece("AVBay",           PressurantBay2.x + PressurantBay2.length, 0,           diameter_int/2, 396,  5,     "cylindrique")
RecoveryBay     = Piece("RecoveryBay",     AVBay.x + AVBay.length,               0,              diameter_int/2, 396,  5,     "cylindrique")
Nose            = Piece("Nose",            RecoveryBay.x + RecoveryBay.length,   0,              diameter_int/2, 1022, 1.6,   "conique")
Gaine           = Piece("Gaine",           LOx.x,                                diameter_int/2, diameter_int/5,
                         LOx.length + MidBay.length + PressurantBay1.length + ETH.length + PressurantBay2.length/2,
                         1, "demi-cylindrique")

rocket = [EngineBay, LOx, MidBay, PressurantBay1, ETH, PressurantBay2, AVBay, RecoveryBay, Nose, Gaine]

In [18]:
def volume_cylindrique(piece):
    return math.pi * piece.radius**2 * piece.length

def volume_conique(piece):
    return math.pi * piece.radius**2 * piece.length / 3

def volume_demi_cylindrique(piece):
    return math.pi * piece.radius**2 * piece.length / 2

VOLUMES = {
    "cylindrique":      volume_cylindrique,
    "conique":          volume_conique,
    "demi-cylindrique": volume_demi_cylindrique,
}

In [22]:
# rayon à la position xi
def _rayon_max(piece, nom, xi):
    if nom == "conique":
        return piece.radius * (1 - xi / piece.length)
    return piece.radius   


def centre_masse_piece(piece: Piece, nom_type: str) -> tuple[float, float]:
    volume = VOLUMES[nom_type](piece)
    rho = piece.mass / volume

    def integrand_x(r, xi):
        R = _rayon_max(piece, nom_type, xi)
        if R <= 0:
            return 0.0
        ang = math.pi if nom_type == "demi-cylindrique" else 2 * math.pi
        return (piece.x + xi) * r * ang

    int_x, _ = dblquad(
        integrand_x,
        0, piece.length,
        0, lambda xi: _rayon_max(piece, nom_type, xi),
    )
    x_cm = rho * int_x / piece.mass

    if nom_type == "demi-cylindrique":
        ANG_INT_Y = 2.0  # ∫₀^π sin(φ) dφ = 2

        def integrand_y_demi(r, xi):
            R = _rayon_max(piece, nom_type, xi)
            if R <= 0:
                return 0.0
            return (piece.y * math.pi + r * ANG_INT_Y) * r

        int_y, _ = dblquad(
            integrand_y_demi,
            0, piece.length,
            0, lambda xi: _rayon_max(piece, nom_type, xi),
        )
        y_cm = rho * int_y / piece.mass
    else:
        y_cm = piece.y

    return x_cm, y_cm


In [23]:
def centre_masse_assemblage(pieces: list[Piece]) -> tuple[float, float]:
    masse_totale = sum(p.mass for p in pieces)
    
    x_num, y_num = 0.0, 0.0
    for piece in pieces:
        xc, yc = centre_masse_piece(piece, piece.forme)
        print(f"  {piece.nom:25s} → x_cm={xc:.4f}, y_cm={yc:.4f}")
        x_num += piece.mass * xc
        y_num += piece.mass * yc

    return x_num / masse_totale, y_num / masse_totale

In [24]:
print("Calcul des centres de masse individuels :")
x_cm, y_cm = centre_masse_assemblage(rocket)
print(f"\nCentre de masse de l'assemblage :")
print(f"  x_cm = {x_cm:.6f}")
print(f"  y_cm = {y_cm:.6f}")


Calcul des centres de masse individuels :
  EngineBay                 → x_cm=497.5000, y_cm=0.0000
  LOx                       → x_cm=1306.0000, y_cm=0.0000
  MidBay                    → x_cm=1757.0000, y_cm=0.0000
  PressurantBay1            → x_cm=2037.0000, y_cm=0.0000
  ETH                       → x_cm=2488.0000, y_cm=0.0000
  PressurantBay2            → x_cm=2939.0000, y_cm=0.0000
  AVBay                     → x_cm=3277.0000, y_cm=0.0000
  RecoveryBay               → x_cm=3673.0000, y_cm=0.0000
  Nose                      → x_cm=4126.5000, y_cm=0.0000
  Gaine                     → x_cm=1967.0000, y_cm=140.3718

Centre de masse de l'assemblage :
  x_cm = 2045.856263
  y_cm = 2.930518
